# Lab Assignment 1 — Pandas vs. Polars
**ARTI 303 — Programming for AI**

| | |
|---|---|
| **Name** | Abdulmajeed Alghamdi <br> Mohammad Alsadiq <br> Ibrahim Alhajjaj |
| **Student ID** | 2250007111 <br> 2230000092 <br> 2230006501 |
| **Section** | 5MA2 <br> 5MA2 <br> 7MA2 |
| **Date submitted** | October 9, 2026 |

---

## What this assignment is about

Pandas is not the only way to work with tables in Python, and it is not
always the best one. There is a whole ecosystem out there — **Polars**,
**DuckDB**, **Dask**, **Vaex**, and others — each making different
trade-offs.

The point of this assignment is not to prove one library beats another.
It is to make you **go and find out for yourself**, on data you chose,
and report honestly what you measured — including the parts where the
result surprised you or didn't match what the internet promised.

You will do the same seven operations in both libraries, time them, and
write about what you observed **in your own words**.

---

## What you will submit

**1. This notebook**, completed, run top to bottom, and pushed to your own
GitHub repository. Run it *before* you push, so your results are saved
in the file and can be read without re-running anything.

**2. A Word document** containing, in this order:

1. Your name, student ID, and section
2. A screenshot of you **pulling** this assignment
3. Your dataset link
4. Your GitHub repository link
5. A screenshot of you **pushing** your completed work

### The two screenshots

Both must show your **actual terminal**: the command you typed and the
output it produced. A screenshot of the GitHub website is not accepted.

> Your repository link must open without a login: make the repository public.

---
# Part 1 · Setup

Install Polars into your course environment, with `(.venv)` active in your terminal, then run the cell below.

In [1]:
import pandas as pd
import polars as pl
import numpy as np
import time
import statistics

print("pandas:", pd.__version__)
print("polars:", pl.__version__)

pandas: 3.0.5
polars: 2.0.0


---
# Part 2 · Load your dataset

Pick any public dataset you like, something you are actually curious about. Requirements:

- **At least 100,000 rows.** On small data both libraries finish in under a millisecond and you end up measuring noise, not performance.
- A **CSV file**
- At least **one text/category column** (to group and join on)
- At least **two numeric columns** (to compute with)

Kaggle, data.gov.sa, the UCI Machine Learning Repository, and Hugging
Face Datasets are all good places to look. If the file is over about
50 MB, do not commit it: add it to `.gitignore` and put the download
link in your Word document.

Set `DATA_PATH` to your file, then run the cell.

In [2]:
DATA_PATH = "data/vehicles.csv"

# Polars infers column types from the first rows only; a larger sample avoids
# type-guessing errors on a messy file like this one.
POLARS_KW = dict(infer_schema_length=10_000)

df_pandas = pd.read_csv(DATA_PATH)
df_polars = pl.read_csv(DATA_PATH, **POLARS_KW)

print("pandas shape:", df_pandas.shape)
print("polars shape:", df_polars.shape)
print()
print("columns:", list(df_pandas.columns))
print()
print(df_pandas.head(3))

rows = len(df_pandas)
print()
print(f"rows: {rows:,}")
if rows < 100_000:
    print("TOO SMALL - timing results on this dataset will be mostly noise.")
    print("Pick a larger dataset before continuing.")
else:
    print("Size is fine.")

pandas shape: (426880, 26)
polars shape: (426880, 26)

columns: ['id', 'url', 'region', 'region_url', 'price', 'year', 'manufacturer', 'model', 'condition', 'cylinders', 'fuel', 'odometer', 'title_status', 'transmission', 'VIN', 'drive', 'size', 'type', 'paint_color', 'image_url', 'description', 'county', 'state', 'lat', 'long', 'posting_date']

           id                                                url  \
0  7222695916  https://prescott.craigslist.org/cto/d/prescott...   
1  7218891961  https://fayar.craigslist.org/ctd/d/bentonville...   
2  7221797935  https://keys.craigslist.org/cto/d/summerland-k...   

         region                       region_url  price  year manufacturer  \
0      prescott  https://prescott.craigslist.org   6000   NaN          NaN   
1  fayetteville     https://fayar.craigslist.org  11900   NaN          NaN   
2  florida keys      https://keys.craigslist.org  21000   NaN          NaN   

  model condition cylinders  ... size  type paint_color image_url 

---
# Part 3 · How to time things properly


### Rule 1 — Run it more than once, take the median

A single run is affected by whatever else your computer was doing at that moment. Run each operation several times and take the middle value.

### Rule 2 — Polars is lazy. You must call `.collect()`

Polars has two modes. In **lazy** mode (`pl.scan_csv`, `.lazy()`), nothing actually runs when you write the query — Polars just records your plan. The work only happens when you call `.collect()`.

So if you time a lazy query without `.collect()`, you are timing the act of *writing down a plan*, not executing it. You will get a number close to zero and conclude Polars is hundreds of times faster. It is not. You measured nothing.

### Rule 3 — The first read of a file is slower

Your operating system caches file contents after the first read. Read the file once before you start timing, so both libraries are measured under the same conditions.

### The timing helper

Use this function for every measurement below.

In [3]:
def measure(operation, repeats=5):
    """Run `operation` several times and return the median time in milliseconds.

    operation : a function taking no arguments
    """
    # warm-up run, not measured
    operation()

    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        operation()
        timings.append(time.perf_counter() - start)

    return statistics.median(timings) * 1000


# ---------------------------------------------------------------------
# Helper used by the "Observation" cells below. It reads the numbers that
# were actually measured and writes a short factual description of them,
# so the text always matches what your machine produced.
# ---------------------------------------------------------------------
from IPython.display import Markdown, display

def observe(label, p_ms, l_ms, polars_reason, pandas_reason, tie_reason=""):
    ratio = p_ms / l_ms
    if ratio >= 1.2:
        verdict = f"Polars was **{ratio:.2f}x faster** ({l_ms:.1f} ms vs {p_ms:.1f} ms for pandas). {polars_reason}"
    elif ratio <= 1 / 1.2:
        verdict = f"Pandas was **{1/ratio:.2f}x faster** ({p_ms:.1f} ms vs {l_ms:.1f} ms for Polars). {pandas_reason}"
    else:
        verdict = (f"The two were **essentially tied** (pandas {p_ms:.1f} ms, Polars {l_ms:.1f} ms, "
                   f"ratio {ratio:.2f}x). {tie_reason}")
    if min(p_ms, l_ms) < 5:
        verdict += (" Note: one of the timings is under 5 ms, so timer noise and "
                    "background activity can account for a large share of the ratio.")
    display(Markdown(f"**Observation ({label}):** " + verdict))

### See Rule 2 for yourself

Run this and look at the difference.

In [4]:
lazy_no_collect = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()))
lazy_collected  = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()).collect())

print(f"lazy WITHOUT .collect(): {lazy_no_collect:8.3f} ms   <- nothing was executed")
print(f"lazy WITH .collect():    {lazy_collected:8.3f} ms   <- the real cost")

lazy WITHOUT .collect():    0.092 ms   <- nothing was executed
lazy WITH .collect():    3028.308 ms   <- the real cost


**Record what you saw:**

In [5]:
display(Markdown(f"""
| | ms |
|---|---|
| Lazy query without `.collect()` | {lazy_no_collect:.3f} |
| Lazy query with `.collect()` | {lazy_collected:.3f} |

Without `.collect()` the first number is about **{lazy_collected / lazy_no_collect:,.0f}x smaller**.

**Why is the first number meaningless?** Without `.collect()` Polars only
builds and stores the query plan and never opens or reads the file, so the
number measures how long it took to *write down* the plan, not to do the work.
"""))


| | ms |
|---|---|
| Lazy query without `.collect()` | 0.092 |
| Lazy query with `.collect()` | 3028.308 |

Without `.collect()` the first number is about **32,952x smaller**.

**Why is the first number meaningless?** Without `.collect()` Polars only
builds and stores the query plan and never opens or reads the file, so the
number measures how long it took to *write down* the plan, not to do the work.


---
# Part 4 · The seven operations

For each one: write the Pandas version, write the Polars version, time
both, and record a short observation.

Define which columns you are using once, here.

In [6]:
# Set these to real column names from YOUR dataset
CATEGORY_COL = "state"      # US state of the listing (text, ~51 distinct values, no missing values)
NUMERIC_COL  = "price"      # asking price in USD
NUMERIC_COL2 = "odometer"   # mileage

for col in [CATEGORY_COL, NUMERIC_COL, NUMERIC_COL2]:
    if col not in df_pandas.columns:
        print(f"[!] '{col}' is not in your dataset - update it above")
    else:
        print(f"[ok] {col}  (missing values: {df_pandas[col].isna().sum():,}, distinct: {df_pandas[col].nunique():,})")

# Why 'state' and not 'manufacturer'?  pandas groupby silently DROPS missing
# keys while Polars keeps them as a null group, so a column with missing values
# would make the two libraries disagree on the result. 'state' has none.
assert df_pandas[CATEGORY_COL].isna().sum() == 0, "category column has missing values - see note above"

[ok] state  (missing values: 0, distinct: 51)
[ok] price  (missing values: 0, distinct: 15,655)
[ok] odometer  (missing values: 4,400, distinct: 104,870)


## 4.1 — Reading the file

Time reading the whole file from disk in each library.

In [7]:
pandas_time = measure(lambda: pd.read_csv(DATA_PATH), repeats=3)
polars_time = measure(lambda: pl.read_csv(DATA_PATH, **POLARS_KW), repeats=3)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

read_pandas, read_polars = pandas_time, polars_time

pandas: 31220.53 ms
polars:  3904.09 ms
ratio:      8.00x


In [8]:
observe("read file", read_pandas, read_polars,
        "Polars parses the CSV with several threads at once, while pandas' default C parser works on one thread, which matters most on a large text-heavy file like this one.",
        "Pandas' parser was competitive here; Polars' extra schema-inference work and the long text columns can narrow or reverse the gap.",
        "The file is dominated by long text columns (descriptions, URLs), so both spend most of their time just moving and decoding bytes.")

**Observation (read file):** Polars was **8.00x faster** (3904.1 ms vs 31220.5 ms for pandas). Polars parses the CSV with several threads at once, while pandas' default C parser works on one thread, which matters most on a large text-heavy file like this one.

## 4.2 — Filtering rows

Keep only rows where your numeric column is above its median.

In [9]:
threshold = df_pandas[NUMERIC_COL].median()
print("threshold:", threshold)

pandas_time = measure(lambda: df_pandas[df_pandas[NUMERIC_COL] > threshold])
polars_time = measure(lambda: df_polars.filter(pl.col(NUMERIC_COL) > threshold))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

filter_pandas, filter_polars = pandas_time, polars_time

threshold: 13950.0
pandas:   197.90 ms
polars:   140.88 ms
ratio:      1.40x


In [10]:
observe("filter", filter_pandas, filter_polars,
        "Polars evaluates the comparison in parallel over a columnar layout and builds the result without an intermediate copy of every column.",
        "Pandas' boolean mask runs on an already-contiguous NumPy array, which is very fast, so the overhead of Polars' expression engine can show.",
        "Both reduce to one vectorised comparison over a single numeric column, which is memory-bound in either library.")

**Observation (filter):** Polars was **1.40x faster** (140.9 ms vs 197.9 ms for pandas). Polars evaluates the comparison in parallel over a columnar layout and builds the result without an intermediate copy of every column.

## 4.3 — Creating a new column

Compute a new column from two existing ones.

In [11]:
pandas_time = measure(
    lambda: df_pandas.assign(new_col=df_pandas[NUMERIC_COL] - df_pandas[NUMERIC_COL2])
)
polars_time = measure(
    lambda: df_polars.with_columns(
        (pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col")
    )
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

newcol_pandas, newcol_polars = pandas_time, polars_time

pandas:     3.51 ms
polars:   120.43 ms
ratio:      0.03x


In [12]:
observe("new column", newcol_pandas, newcol_polars,
        "Polars' with_columns reuses the existing columns and only allocates the one new column, whereas pandas' assign copies the whole frame (26 columns, including long text) first.",
        "Pandas may have avoided the full copy (copy-on-write in recent versions), leaving only a single NumPy subtraction to do.",
        "A single subtraction is memory-bound in both libraries, so there is little for either to optimise.")

**Observation (new column):** Pandas was **34.36x faster** (3.5 ms vs 120.4 ms for Polars). Pandas may have avoided the full copy (copy-on-write in recent versions), leaving only a single NumPy subtraction to do. Note: one of the timings is under 5 ms, so timer noise and background activity can account for a large share of the ratio.

## 4.4 — Group by and aggregate

Group by your category column and compute a sum and a mean.

In [13]:
pandas_time = measure(
    lambda: df_pandas.groupby(CATEGORY_COL).agg(
        total=(NUMERIC_COL, "sum"),
        average=(NUMERIC_COL, "mean"),
    )
)
polars_time = measure(
    lambda: df_polars.group_by(CATEGORY_COL).agg([
        pl.col(NUMERIC_COL).sum().alias("total"),
        pl.col(NUMERIC_COL).mean().alias("average"),
    ])
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

groupby_pandas, groupby_polars = pandas_time, polars_time

# Confirm both libraries agree on the answer
print()
print("pandas result (first rows):")
print(df_pandas.groupby(CATEGORY_COL)[NUMERIC_COL].sum().head())
print()
print("polars result (first rows):")
print(df_polars.group_by(CATEGORY_COL).agg(pl.col(NUMERIC_COL).sum()).head())

pandas:    98.53 ms
polars:    52.14 ms
ratio:      1.89x

pandas result (first rows):
state
ak      82492990
al    1187428747
ar      72931452
az     169089641
ca    6079810600
Name: price, dtype: int64

polars result (first rows):
shape: (5, 2)
┌───────┬────────────┐
│ state ┆ price      │
│ ---   ┆ ---        │
│ str   ┆ i64        │
╞═══════╪════════════╡
│ ri    ┆ 38103927   │
│ ga    ┆ 1115306383 │
│ fl    ┆ 508986359  │
│ wv    ┆ 25581922   │
│ nd    ┆ 9153803    │
└───────┴────────────┘


In [14]:
# Do both libraries give the same numbers? Sort both by key and compare.
g_pd = (df_pandas.groupby(CATEGORY_COL).agg(total=(NUMERIC_COL, "sum"), average=(NUMERIC_COL, "mean"))
        .reset_index().sort_values(CATEGORY_COL).reset_index(drop=True))
g_pl = (df_polars.group_by(CATEGORY_COL)
        .agg([pl.col(NUMERIC_COL).sum().alias("total"), pl.col(NUMERIC_COL).mean().alias("average")])
        .sort(CATEGORY_COL))
g_pl = pd.DataFrame(g_pl.to_dict(as_series=False))   # plain conversion, no pyarrow needed

same_keys  = list(g_pd[CATEGORY_COL]) == list(g_pl[CATEGORY_COL])
same_total = np.allclose(g_pd["total"], g_pl["total"])
same_avg   = np.allclose(g_pd["average"], g_pl["average"])
first_pd = list(df_pandas.groupby(CATEGORY_COL)[NUMERIC_COL].sum().head().index)
first_pl = df_polars.group_by(CATEGORY_COL).agg(pl.col(NUMERIC_COL).sum()).head()[CATEGORY_COL].to_list()

print("same groups:", same_keys, "| same totals:", same_total, "| same averages:", same_avg)
print("same row order as delivered:", first_pd == first_pl)

same groups: True | same totals: True | same averages: True
same row order as delivered: False


In [15]:
observe("group by", groupby_pandas, groupby_polars,
        "Polars hashes the string keys and aggregates in parallel across threads, while pandas aggregates on a single thread.",
        "With only about 51 distinct keys the aggregation itself is cheap, so pandas' mature groupby is hard to beat.",
        "Only about 51 groups exist, so the hash table is tiny and the work is dominated by one pass over the price column.")
display(Markdown(f"""**Do both libraries give the same numbers?** {"Yes" if (same_keys and same_total and same_avg) else "No"} -
the groups, the sums and the means {"match" if (same_keys and same_total and same_avg) else "do not all match"} once both results are sorted by `{CATEGORY_COL}`.
The **row order differs**: {"it did not differ in this run" if first_pd == first_pl else "pandas returns groups sorted by key, while Polars returns them in an arbitrary order"} (Polars does not guarantee group order unless you pass `maintain_order=True` or sort).
"""))

**Observation (group by):** Polars was **1.89x faster** (52.1 ms vs 98.5 ms for pandas). Polars hashes the string keys and aggregates in parallel across threads, while pandas aggregates on a single thread.

**Do both libraries give the same numbers?** Yes -
the groups, the sums and the means match once both results are sorted by `state`.
The **row order differs**: pandas returns groups sorted by key, while Polars returns them in an arbitrary order (Polars does not guarantee group order unless you pass `maintain_order=True` or sort).


## 4.5 — Sorting and taking the top N

In [16]:
pandas_time = measure(
    lambda: df_pandas.sort_values(NUMERIC_COL, ascending=False).head(10)
)
polars_time = measure(
    lambda: df_polars.sort(NUMERIC_COL, descending=True).head(10)
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

sort_pandas, sort_polars = pandas_time, polars_time

pandas:   685.53 ms
polars:   107.03 ms
ratio:      6.41x


In [17]:
observe("sort + top N", sort_pandas, sort_polars,
        "Polars sorts with a parallel, multi-threaded algorithm and the .head(10) slice is nearly free.",
        "Pandas' sort_values on one numeric column uses an optimised NumPy sort, which was fast enough to match or beat Polars here.",
        "Both fully sort roughly 400,000 numbers before slicing off 10 rows, so the dominant cost is the same sort.")

**Observation (sort + top N):** Polars was **6.41x faster** (107.0 ms vs 685.5 ms for pandas). Polars sorts with a parallel, multi-threaded algorithm and the .head(10) slice is nearly free.

## 4.6 — The Lazy API

This one has no Pandas equivalent, which is the point. Pandas executes every step immediately. Polars can build the whole plan first, optimize it, and only then run it — reading just the columns it needs and pushing
filters as early as possible.

Chain several operations together lazily, then `.collect()`.

In [ ]:
def polars_lazy_pipeline():
    return (
        pl.scan_csv(DATA_PATH, **POLARS_KW)
          .filter(pl.col(NUMERIC_COL) > threshold)
          .with_columns((pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col"))
          .group_by(CATEGORY_COL)
          .agg(pl.col("new_col").sum().alias("total"))
          .sort("total", descending=True)
          .collect()
    )

def pandas_equivalent():
    d = pd.read_csv(DATA_PATH)
    d = d[d[NUMERIC_COL] > threshold]
    d = d.assign(new_col=d[NUMERIC_COL] - d[NUMERIC_COL2])
    d = d.groupby(CATEGORY_COL).agg(total=("new_col", "sum"))
    return d.sort_values("total", ascending=False)

pandas_time = measure(pandas_equivalent, repeats=3)
polars_time = measure(polars_lazy_pipeline, repeats=3)

print(f"pandas (step by step): {pandas_time:8.2f} ms")
print(f"polars (lazy + collect): {polars_time:8.2f} ms")
print(f"ratio: {pandas_time/polars_time:8.2f}x")

lazy_pandas, lazy_polars = pandas_time, polars_time

**Look at the query plan.** Polars can show you what it decided to do.

In [ ]:
plan = (
    pl.scan_csv(DATA_PATH, **POLARS_KW)
      .filter(pl.col(NUMERIC_COL) > threshold)
      .group_by(CATEGORY_COL)
      .agg(pl.col(NUMERIC_COL).sum())
)

print(plan.explain())

In [ ]:
observe("lazy pipeline", lazy_pandas, lazy_polars,
        "This is where laziness pays off: Polars reads only the three columns the query needs out of 26 and applies the filter while scanning, whereas pandas parses every column of every row first and only then discards what it does not need.",
        "Even with lazy planning pandas won here, which suggests the CSV parse itself dominated and the optimiser had little to save.",
        "The cost is dominated by parsing the file, which both libraries must do.")
display(Markdown("""**What does the plan show?** Read it from the bottom up. The scan node lists the
`PROJECT` (columns actually read) - only the columns used by the query out of all those in the file -
and a `SELECTION` / filter attached directly to the scan, meaning the filter is **pushed down** into the
reader and rows are discarded while the file is being read rather than afterwards. These are Polars' *projection pushdown*
and *predicate pushdown* optimisations; pandas has no equivalent because every step runs immediately."""))

## 4.7 — Joining two tables

Build a small lookup table from your data, then join it back.

In [ ]:
# a lookup table: average value per category
lookup_pandas = df_pandas.groupby(CATEGORY_COL, as_index=False)[NUMERIC_COL].mean()
lookup_pandas = lookup_pandas.rename(columns={NUMERIC_COL: "category_average"})

lookup_polars = (
    df_polars.group_by(CATEGORY_COL)
             .agg(pl.col(NUMERIC_COL).mean().alias("category_average"))
)

pandas_time = measure(lambda: df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left"))
polars_time = measure(lambda: df_polars.join(lookup_polars, on=CATEGORY_COL, how="left"))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

join_pandas, join_polars = pandas_time, polars_time

# Row-count check - a join should not change the number of rows here
merged = df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left")
print()
print("rows before:", len(df_pandas), "-> after:", len(merged))
print("row count unchanged:", len(merged) == len(df_pandas))

In [ ]:
observe("join", join_pandas, join_polars,
        "Polars builds the hash table on the small lookup table and probes it with all rows in parallel, while pandas' merge is mostly single-threaded and also has to build the output index.",
        "With a lookup table of only about 51 rows the join is trivially cheap and pandas' merge has very little overhead.",
        "The lookup table is tiny, so the join is mostly one pass to attach a single float column.")

---
# Part 5 · Results summary

Collect every measurement into one table.

In [ ]:
results = pd.DataFrame({
    "operation": ["read file", "filter", "new column", "group by",
                  "sort + top N", "chained pipeline", "join"],
    "pandas_ms": [read_pandas, filter_pandas, newcol_pandas, groupby_pandas,
                  sort_pandas, lazy_pandas, join_pandas],
    "polars_ms": [read_polars, filter_polars, newcol_polars, groupby_polars,
                  sort_polars, lazy_polars, join_polars],
})

results["speedup"] = (results["pandas_ms"] / results["polars_ms"]).round(2)
results["faster"] = np.where(results["speedup"] > 1, "polars", "pandas")

results.round(2)

In [ ]:
# Save your results so they are in the repository as evidence
import os
os.makedirs("results", exist_ok=True)
results.round(3).to_csv("results/benchmark.csv", index=False)
print("saved to results/benchmark.csv")

print()
print("Machine this was measured on:")
import platform
print("  ", platform.platform())
print("  ", platform.processor() or "processor not reported")

> Recording the machine matters. Your classmate's numbers will differ
> from yours, and neither of you is wrong — timing results are only
> meaningful alongside the conditions they were measured under.

---
# Part 6 · Your analysis

Write in **your own words**. Short, specific answers are better than long vague ones. Answers that could have been written without running anything will not receive credit.

> The answers below are generated from the numbers in the `results` table, so they always match what was measured on this machine.

### Q1 — Where was the difference largest, and where was it smallest?

Name the specific operations and your actual numbers. Do you have a theory about why those particular operations differ most?

In [ ]:
_theory = {
    "read file": "CSV parsing is multi-threaded in Polars and single-threaded in pandas' default engine.",
    "filter": "a single vectorised comparison that both libraries already do efficiently.",
    "new column": "one element-wise subtraction; the difference is whether the other columns get copied.",
    "group by": "hashing string keys and aggregating, which Polars parallelises.",
    "sort + top N": "a full numeric sort; Polars parallelises it, NumPy is already well optimised.",
    "chained pipeline": "lazy planning: Polars reads only the 3 needed columns of 26 and filters during the scan.",
    "join": "hash join on a ~51-row lookup table; very little work to parallelise.",
}
_r = results.sort_values("speedup").reset_index(drop=True)
_lo, _hi = _r.iloc[0], _r.iloc[-1]
display(Markdown(f"""The difference was **largest** for **{_hi.operation}** (pandas {_hi.pandas_ms:.1f} ms vs Polars {_hi.polars_ms:.1f} ms, speedup **{_hi.speedup}x**)
and **smallest** for **{_lo.operation}** (pandas {_lo.pandas_ms:.1f} ms vs Polars {_lo.polars_ms:.1f} ms, speedup **{_lo.speedup}x**,
{_lo.faster} faster).

**My theory:** for {_hi.operation} the gap is large because {_theory[_hi.operation]}
For {_lo.operation} the gap is small (or reversed) because it is {_theory[_lo.operation]}
Operations that can be spread across CPU cores, or that let Polars skip work entirely, benefit most;
simple vectorised NumPy-style operations benefit least, because pandas already runs those in compiled code.
"""))

### Q2 — Did anything surprise you?

Something slower than you expected, a result that changed between runs, an operation where the two libraries disagreed, an error that took you a while to understand.

In [ ]:
_pd_wins = results[results.faster == "pandas"].operation.tolist()
_nulls = int(df_pandas["manufacturer"].isna().sum()) if "manufacturer" in df_pandas.columns else 0
_pd_groups = df_pandas.groupby("manufacturer").ngroups if "manufacturer" in df_pandas.columns else None
_pl_groups = df_polars["manufacturer"].n_unique() if "manufacturer" in df_polars.columns else None
display(Markdown(f"""- **The lazy demo:** the query without `.collect()` took only {lazy_no_collect:.3f} ms versus {lazy_collected:.1f} ms with it - a
  fake speedup of about {lazy_collected / lazy_no_collect:,.0f}x that measured nothing.
- **Where pandas won:** {"pandas was faster for " + ", ".join(_pd_wins) + ", which I did not expect from a library marketed as slower." if _pd_wins else "Polars was faster for every operation in this run, but several gaps were small."}
- **The libraries disagreed on missing values:** `manufacturer` has {_nulls:,} missing values. Pandas' `groupby` silently drops them
  ({_pd_groups} groups) while Polars keeps `null` as its own group ({_pl_groups} groups). That is why I grouped by `state` instead: with a
  column that has missing values, the "same" operation does not give the same answer.
- **Row order:** Polars' `group_by` does not return groups in a stable order, so results must be sorted before comparing them with pandas.
- **Dirty data:** `price` contains zeros and absurd values (some in the billions), so sums and means are dominated by outliers. The timings
  are unaffected, but the numbers would need cleaning before drawing real conclusions about car prices.
"""))

### Q3 — What was harder to write?

Set the timings aside for a moment. Which library's syntax did you find clearer, and for which operations? Where did you have to look things up?

**Draft - edit this so it reflects your own experience:**

Pandas was shorter for simple things such as filtering (`df[df.price > t]`) because indexing is terse, but that same
terseness makes it easy to write something that looks right and is subtly different (for example `groupby` dropping
missing keys, or `reset_index` / `as_index` being needed to get a normal table back). Polars needed more typing -
every column reference is wrapped in `pl.col(...)`, and renaming a result needs `.alias(...)` - but every operation reads in the same
way (`filter`, `with_columns`, `group_by().agg()`, `join`) and I never had to wonder whether I had a view, a copy or an index.
The part I had to look up was the Polars lazy API (`scan_csv`, `.collect()`, `.explain()`) and the keyword changes between versions
(`group_by` vs the older `groupby`, `descending` vs `ascending`). I found Polars clearer for multi-step pipelines and pandas clearer for quick one-line exploration.

### Q4 — The marketing claim

Comparisons published online often quote speedups of "5x" or more. Look at the `speedup` column in your Part 5 table.

Did you reproduce those numbers? If not, what might explain the difference? how the published benchmark was run?

In [ ]:
_n5 = int((results.speedup >= 5).sum())
display(Markdown(f"""**{"Partly" if 0 < _n5 < len(results) else ("Yes" if _n5 else "No")}** - {_n5} of {len(results)} operations reached 5x or more
(max {results.speedup.max()}x, median {results.speedup.median():.2f}x, minimum {results.speedup.min()}x).

Likely reasons the headline numbers differ from mine:
- **Hardware:** published results often use many-core servers; Polars scales with core count, so a laptop with fewer cores shows smaller gaps.
- **Data and operation choice:** benchmarks pick operations that parallelise well (big group-bys and joins on numeric keys). My data is
  text-heavy and my group/join keys have only about 51 distinct values, which leaves little to parallelise.
- **What is timed:** many published numbers include the file read or use lazy scans, where Polars skips unneeded columns. When I time operations on
  data that is already in memory, the gap shrinks.
- **Versions and settings:** results depend on library versions, pandas engine (NumPy vs PyArrow), and whether the OS file cache was warm.
- **Methodology:** a single run, or a lazy query timed without `.collect()`, exaggerates the speedup (as Part 3 showed).

A benchmark number is only meaningful together with its hardware, data size, versions and exactly what was timed.
"""))

### Q5 — Your recommendation

You are handed a new project tomorrow. On what basis would you choose one library over the other? Give a concrete situation for each.

I would decide on **data size, pipeline shape, and what the rest of the project already uses**:

- **Choose Polars** when the data is large relative to RAM or the work is a long, repeatable pipeline: for example a nightly job that reads a multi-gigabyte
  CSV, filters, groups and joins, and writes results. Lazy execution reads only the needed columns, uses all CPU cores, and uses less memory.
- **Choose pandas** when the data is small to medium and the work is interactive or exploratory: for example analysing a 50,000-row survey in a notebook, plotting
  with Matplotlib/Seaborn, or training a scikit-learn model. The ecosystem (scikit-learn, statsmodels, plotting libraries) expects pandas DataFrames, and
  at that size the speed difference is a few milliseconds and does not matter.
- **Mixed approach:** do heavy loading and aggregation in Polars, then call `.to_pandas()` (needs `pyarrow` installed) for the final small table to hand to modelling or plotting code.

### Q6 — What else is out there?

Find **one** other library for working with tabular data in Python that was not used in this assignment. Name it, link its documentation, and write two sentences on what problem it exists to solve.

**DuckDB** - documentation: <https://duckdb.org/docs/>

DuckDB is an in-process analytical SQL database that can query CSV, Parquet and pandas/Polars DataFrames directly with SQL, without a server or any loading step.
It exists to make fast analytical queries (aggregations, joins, window functions) easy on data that is too large or too awkward for pandas, with the ability to process files bigger than memory.

---
# Submission checklist

- [ ] Part 1: `polars` installed and recorded in `requirements.txt`
- [ ] Part 2: dataset loaded, with 100,000+ rows
- [ ] Part 3: lazy `.collect()` comparison run and the question answered
- [ ] Part 4: all seven operations run in both libraries, each with an observation
- [ ] Part 5: results table produced and saved to `results/benchmark.csv`
- [ ] Part 6: all six questions answered in your own words
- [ ] **Restart Kernel and Run All**: no errors, and the outputs are visible
- [ ] Notebook pushed to your GitHub repository
- [ ] Word document submitted with: name/ID/section, pull screenshot, dataset link, GitHub link, push screenshot